# Grade 9 · Unit 3 — SQL on district data

> **SYNTHETIC data:** the numbers are invented. District names such as `Hill-07` are fictional. Use the real table later (`python tools/fetch_data.py census-districts`).

We follow **PRIMM**: *Predict → Run → Investigate → Modify → Make*. Write your prediction before you run each cell.

In [ ]:
from pathlib import Path
import pandas as pd

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

import sqlite3

districts = pd.read_csv(data_file("census-districts", "districts_synthetic.csv"))
schools = pd.read_csv(data_file("census-districts", "schools_synthetic.csv"))

con = sqlite3.connect(":memory:")          # a database that lives only in memory
districts.to_sql("districts", con, index=False)
schools.to_sql("schools", con, index=False)

def q(sql):
    """Run a SQL query and show the result as a table."""
    return pd.read_sql_query(sql, con)

q("SELECT name FROM sqlite_master WHERE type='table'")

## 1. SELECT and COUNT
**Predict:** how many rows will the first query return? What will the second show?

In [ ]:
print(len(q("SELECT * FROM districts")))
q("SELECT district, ecological_belt, population FROM districts LIMIT 5")

## 2. WHERE and ORDER BY
**Investigate:** which clause keeps only Terai rows? Which one puts the biggest first?

In [ ]:
q("""
SELECT district, population, literacy_pct
FROM districts
WHERE ecological_belt = 'Terai'
ORDER BY population DESC
LIMIT 5
""")

**Modify:** change `'Terai'` to `'Mountain'`; change `DESC` to `ASC`; add `AND literacy_pct > 70`.

## 3. GROUP BY
**Predict:** how many rows will this return? (Hint: how many belts are there?)

In [ ]:
q("""
SELECT ecological_belt,
       COUNT(*)                  AS n_districts,
       SUM(population)           AS total_population,
       ROUND(AVG(literacy_pct),1) AS avg_literacy
FROM districts
GROUP BY ecological_belt
ORDER BY total_population DESC
""")

## 4. JOIN — combining two tables
The `schools` table has one row per district too. We match rows with the shared **key** `district_id`.

In [ ]:
q("""
SELECT d.district, d.population, s.n_secondary_schools,
       ROUND(1.0 * d.population / s.n_secondary_schools) AS people_per_school
FROM districts d
JOIN schools s ON d.district_id = s.district_id
ORDER BY people_per_school DESC
LIMIT 5
""")

## 5. A rate, not a count
Girls' share of enrolment by district — then which belt has the lowest average?

In [ ]:
q("""
SELECT d.ecological_belt,
       ROUND(100.0 * SUM(s.enrolment_girls) / SUM(s.enrolment_girls + s.enrolment_boys), 1) AS girls_pct
FROM districts d JOIN schools s ON d.district_id = s.district_id
GROUP BY d.ecological_belt
ORDER BY girls_pct
""")

## Make
Write your own query for this question, then explain the result in one sentence **including one limit** (this table is synthetic):

> *Which belt has the most people living abroad per 1,000 people (`absentee_population`)?*

Use the code cell below.

In [ ]:
answer = q("""
SELECT ecological_belt,
       ROUND(1000.0 * SUM(absentee_population) / SUM(population), 1) AS abroad_per_1000
FROM districts
GROUP BY ecological_belt
ORDER BY abroad_per_1000 DESC
""")
assert len(answer) == 3 and len(districts) == 77
answer